# Week 2 — Baseline Agent Comparison

Comparing Random, Fixed Price, Time-Based Discount, and Demand-Based agents.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
from pricing_env import PricingEnv
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent, DemandBasedAgent

def run_episodes(agent, env, n_episodes=100, has_reset=False):
    revenues = []
    sell_through_rates = []
    
    for _ in range(n_episodes):
        obs, info = env.reset()
        if has_reset:
            agent.reset()
        total_reward = 0
        initial_inventory = env.max_inventory
        
        done = False
        while not done:
            action = agent.act(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        
        revenues.append(total_reward)
        remaining_inventory = obs[0]
        sell_through = (initial_inventory - remaining_inventory) / initial_inventory
        sell_through_rates.append(sell_through)
    
    return {
        "mean_revenue": np.mean(revenues),
        "std_revenue": np.std(revenues),
        "mean_sell_through": np.mean(sell_through_rates)
    }

In [ ]:
env = PricingEnv()

agents = {
    "FixedPrice": (FixedPriceAgent(), False),
    "TimeBasedDiscount": (TimeBasedDiscountAgent(), True),
    "DemandBased": (DemandBasedAgent(), False),
}

results = []
for name, (agent, has_reset) in agents.items():
    stats = run_episodes(agent, env, n_episodes=100, has_reset=has_reset)
    results.append({
        "Agent": name,
        "Episodes": 100,
        "Mean Revenue": round(stats["mean_revenue"], 2),
        "Std Dev": round(stats["std_revenue"], 2),
        "Sell-Through Rate": f"{stats['mean_sell_through']*100:.1f}%"
    })

df = pd.DataFrame(results)
df